# Citi Bike Extended Analysis — Skeleton Notebook

This notebook walks you through an **extended** version of the Citi Bike
visualization project. Every section explains *what* to do and *why* — you
write the code. If you get stuck, the companion `Citibike_Extended_Solution.ipynb`
has a full worked version, and `Citibike_Cheatsheet.ipynb` has every function
you'll need with mini-examples.

Data file expected in the same folder: `january_trips_subset.csv`
(or `january_trips.csv` for the full month).

## 0. Setup

Load the packages you'll need for this whole notebook: `dplyr` (data wrangling),
`ggplot2` (plotting), `geosphere` (distance calculation), and `lubridate`
(date/time parsing, needed for the new temporal-pattern section).

In [ ]:
# install.packages(c("dplyr", "ggplot2", "geosphere", "lubridate"))

# Load your libraries here

## 1. Data Audit (new)

Before trusting any chart, sanity-check the raw data. Load `january_trips_subset.csv`
into `all_data`, then:

1. Print `str(all_data)` or `head(all_data)` to confirm columns and types.
2. Count missing values per column (hint: `colSums(is.na(all_data))`).
3. Check for impossible values: `tripduration <= 0`, `birth.year` implying an
   age over 100 or under 10, and any station with latitude/longitude equal to 0.
4. Write one or two sentences (in a markdown cell) summarizing what you found
   and what you plan to do about it (e.g., "I will filter out ages > 100 later").

This step matters: the original project silently discovers some of these issues
later (e.g. ages over 100) — here you find and document them up front.

In [ ]:
# Load the data

In [ ]:
# Structure / types

In [ ]:
# Missing values per column

In [ ]:
# Implausible values: durations, ages, zero coordinates

**Your data audit notes:** *(replace this text)*

## 2. Core EDA — Spatial Heat Map

Make a heat map of start-station locations using `ggplot()` + `geom_bin2d()`.
Use a small bin width on both axes (try `0.001`) so the shapes of Manhattan,
Brooklyn, and Queens emerge. Look for the empty rectangle in Manhattan — that's
Central Park.

In [ ]:
# Heat map of start.station.longitude / start.station.latitude

## 3. Feature Engineering: Subset, Age, Distance, Speed

1. Create `short_trips` by filtering `all_data` to `tripduration < 900` seconds
   (15 minutes) — this keeps the rest of the notebook fast. Using the full
   dataset is fine too if you don't mind longer runtimes.
2. Add an `age` column: `2020 - birth.year`.
3. Add a `distance` column using `geosphere::distHaversine()` on the start and
   end lat/lon pairs (select the two lon/lat columns into `starting_stations`
   and `ending_stations` first).
4. Add a `speed` column: `distance / tripduration` (meters per second).

Print `head()` after each step to confirm it worked before moving on.

In [ ]:
# short_trips <- ...

In [ ]:
# age column

In [ ]:
# distance column (geosphere::distHaversine)

In [ ]:
# speed column

## 4. Speed by Age

1. Group `short_trips` by `age`, summarize the mean `speed` into
   `average_speed_by_age`.
2. Plot a line chart: age on x, mean speed on y.
3. Filter out `age >= 80` (implausible) and redraw.
4. Add a centered title and axis labels.

In [ ]:
# group_by(age) %>% summarize(mean_speed = ...)

In [ ]:
# line plot

In [ ]:
# filter age < 80 and redraw with title + labels

## 5. Speed by Age and Gender

1. Repeat the grouping, this time by `age` AND `gender`, into
   `average_speed_by_age_and_gender`.
2. Plot the line chart again with `color = gender`, filtered to `age < 80`.
3. Convert `gender` to a factor with `as.factor()` inside `mutate()` and redraw
   — this fixes the continuous-color-scale problem.
4. Filter to only genders `1` and `2` (drop unspecified) and relabel the legend
   with `scale_color_discrete(name=..., labels=c(...))` as "Male Identifying" /
   "Female Identifying".

In [ ]:
# group_by(age, gender)

In [ ]:
# line plot colored by gender (raw, before factor fix)

In [ ]:
# mutate gender to factor, redraw

In [ ]:
# filter to gender 1/2, relabel legend, final version

## 6. Stacked Bar Plot of Ages by Gender

1. From `short_trips`, `group_by(age, gender) %>% tally()` into `age_counts`.
2. Plot a stacked bar chart: x = age, y = n, fill = gender, using `geom_col()`.
3. Clean it up: cast gender to a factor, filter to `age < 80` and genders 1/2,
   add a title and relabeled legend.

In [ ]:
# age_counts <- ...

In [ ]:
# raw stacked bar plot

In [ ]:
# cleaned-up final version

## 7. Temporal Patterns (new)

The original project never looks at *time*. Use `lubridate` to unlock it.

1. Parse `starttime` into a proper datetime column (check the actual string
   format first with a `head()` — Citi Bike data is usually
   `"YYYY-MM-DD HH:MM:SS"`).
2. Derive `hour` (0–23), `weekday` (name of day), and a boolean/factor
   `is_weekend`.
3. Plot trip counts by `hour`, as a single line and again faceted by
   `is_weekend` (`facet_wrap(~is_weekend)`).
4. In a markdown cell, describe the shape you see: does the weekday chart show
   two commute peaks (morning + evening)? Does the weekend chart look flatter,
   single-peaked, or shifted later?

In [ ]:
# parse starttime, derive hour / weekday / is_weekend

In [ ]:
# trip counts by hour

In [ ]:
# faceted by weekday vs weekend

**Your interpretation:** *(replace this text)*

## 8. Subscriber vs. Customer (new)

Citi Bike's `usertype` column distinguishes annual **Subscribers** from
pay-per-ride **Customers**. These two groups usually behave very differently.

1. Compare mean `tripduration` and mean `speed` by `usertype`
   (`group_by(usertype) %>% summarize(...)`).
2. Overlay the hour-of-day distribution for each `usertype` on one plot
   (`color = usertype` in `geom_freqpoly()` or `geom_line()` after counting).
3. Write one paragraph: which group looks more "commuter-like" and which looks
   more "leisure-like," and what evidence from the chart supports that?

In [ ]:
# mean duration/speed by usertype

In [ ]:
# hour-of-day distribution by usertype

**Your interpretation:** *(replace this text)*

## 9. Station-Level Flow / Rebalancing (new)

Operations teams need to know which stations run out of bikes (too many
departures) or overflow with docked bikes (too many arrivals).

1. Count departures per station: `group_by(start.station.name) %>% tally()`.
2. Count arrivals per station: `group_by(end.station.name) %>% tally()`.
3. Join the two counts on station name (`full_join`), fill any missing counts
   with 0, and compute `net_flow = arrivals - departures`.
4. Sort and show the top 10 stations that "drain" (most negative net flow) and
   the top 10 that "flood" (most positive net flow). A horizontal bar chart
   (`geom_col() + coord_flip()`) works well for both.

In [ ]:
# departures per station

In [ ]:
# arrivals per station

In [ ]:
# join, net_flow, top drains / floods

In [ ]:
# bar chart of top drain / flood stations

## 10. Distance-Method Sensitivity (new)

The original project flags its own weak point: Haversine gives straight-line
distance, not actual riding distance. You won't call an external API here, but
you can still stress-test the assumption.

1. Create a second speed estimate `speed_adjusted` that divides `distance` by
   `tripduration` **after** multiplying `distance` by a detour factor (a common
   rule of thumb for urban street grids is ~1.3–1.4×, meaning real riding
   distance is 30–40% longer than the straight line).
2. Recompute `average_speed_by_age` using `speed_adjusted` and re-plot it next
   to the original.
3. In a markdown cell: does the *shape* of the age-vs-speed relationship change,
   or only the scale? That distinction tells you whether the original
   conclusion ("younger riders are faster") is robust to the distance
   assumption.

In [ ]:
# speed_adjusted with a detour factor

In [ ]:
# recompute and re-plot average speed by age using speed_adjusted

**Your interpretation:** *(replace this text)*

## 11. Weather Overlay (optional, new)

If you'd like to go further: download a small daily weather summary for NYC,
January 2020 (date, avg temperature, precipitation) from a free source such as
NOAA's Climate Data Online or Visual Crossing, save it as `nyc_weather_jan2020.csv`,
and join it to your trip data by date.

1. Parse the trip date from `starttime` (already have this from step 7).
2. Join daily trip counts (or average trip duration) to the weather table by date.
3. Plot trips-per-day against temperature (scatter + smoothing line, or a
   dual-axis-style comparison using two stacked line plots).
4. Note any relationship you see, and any confound you can't rule out (e.g.,
   weekday/weekend mix changing which days are warm vs. cold).

In [ ]:
# join weather table by date (optional)

In [ ]:
# trips vs temperature

## 12. Recommendation Memo (new)

Write a short memo (3–4 short paragraphs, in this markdown cell) addressed to
Citi Bike's operations team. Structure it as:

1. **Headline finding** — the single most decision-relevant thing you found.
2. **Supporting evidence** — 2–3 numbers or charts from above that back it up.
3. **Caveats** — what assumption (distance method, subset size, missing
   demographic data) most limits your confidence.
4. **Next step** — one concrete thing you'd want (more data, a different
   join, a follow-up analysis) to raise your confidence.

**Your memo:** *(replace this text)*